<a href="https://colab.research.google.com/github/Sathwika2202/NLP/blob/main/Hugging_Face_Tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Hugging Face is an AI company and open-source platform that provides tools and pre-trained models for Natural Language Processing (NLP), computer vision, and machine learning.**

**It allows you to use powerful AI models (like BERT, GPT) with just a few lines of code.**

1. 🤖 Transformers Library

Main library: Transformers (Hugging Face library)

Provides pre-trained models like:
BERT
GPT

Tasks supported:
Text classification
Translation
Question answering
Summarization

2. 📚 Model Hub
A huge collection of pre-trained models

Thousands of models shared by researchers & developers
You can:

Download models

Upload your own models

3. 📊 Datasets Library

Provides ready-to-use datasets


**This installs:**

Transformers (Hugging Face library)
Hugging Face datasets library

In [6]:
!pip install transformers datasets

# **Import Libraries**

In [7]:
from transformers import pipeline

# **Sentiment Analysis**

This line automatically:

Downloads model
Loads tokenizer
Initializes pipeline

In [8]:
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english", revision="714eb0f")

result = classifier("Hugging Face is very easy to use!")
print(result)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9988467693328857}]


# **Dataset Importing**

In [9]:
from datasets import load_dataset
dataset = load_dataset("imdb")
print(dataset)

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [10]:
import pandas as pd
from datasets import load_dataset

# Load the imdb dataset from Hugging Face datasets library
imdb_dataset = load_dataset("imdb")

# Convert the 'train' split to a pandas DataFrame for compatibility
df = imdb_dataset["train"].to_pandas()

print(df.head())

                                                text  label
0  I rented I AM CURIOUS-YELLOW from my video sto...      0
1  "I Am Curious: Yellow" is a risible and preten...      0
2  If only to avoid making this type of film in t...      0
3  This film was probably inspired by Godard's Ma...      0
4  Oh, brother...after hearing about this ridicul...      0


# **Part 2: Using BERT Model**

**Load Tokenizer & Model**

In [11]:
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


**Tokenization**

In [12]:
text = "Transformers are powerful models"

inputs = tokenizer(text,return_tensors="pt")
print(inputs)

{'input_ids': tensor([[  101, 19081,  2024,  3928,  4275,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]])}


In [13]:
outputs = model(**inputs)

# CLS token embedding
cls_embedding = outputs.last_hidden_state[:, 0, :]
print(cls_embedding.shape)

torch.Size([1, 768])


# **Text Classification Using Hugging Face With Sample Data**

**Text → Tokenizer → BERT → CLS Embedding → Naive Bayes → Prediction**

In [14]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split

In [15]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
texts = [
    "I love this product",
    "This is amazing",
    "I am very happy",
    "I hate this",
    "This is bad",
    "Very disappointing"
]

labels = [1, 1, 1, 0, 0, 0]

In [17]:
def get_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    outputs = model(**inputs)

    # CLS embedding
    cls_embedding = outputs.last_hidden_state[:, 0, :]

    return cls_embedding.detach().numpy()[0]

In [18]:
X = np.array([get_embedding(t) for t in texts])
y = np.array(labels)

print("Feature shape:", X.shape)

Feature shape: (6, 768)


In [19]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [20]:
clf = GaussianNB()
clf.fit(X_train, y_train)

GaussianNB()

In [21]:
y_pred = clf.predict(X_test)

print("Predictions:", y_pred)
print("Actual:", y_test)

Predictions: [1 1]
Actual: [0 0]


In [22]:

test_text = "This product is really good"

test_emb = get_embedding(test_text).reshape(1, -1)

prediction = clf.predict(test_emb)

print("Prediction:", "Positive" if prediction[0] == 1 else "Negative")

Prediction: Positive
